# Gradient Masking: Learn Which Parameters to Update

<a target="_blank" href="https://colab.research.google.com/github/dtch1997/maml-inductive-biases/blob/master/maml-sprint-4/01_gradient_mask.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook learns a **binary gradient mask** that controls which LoRA parameters get updated during finetuning. The mask is meta-learned to achieve selective learning: allow Spanish, block CAPS.

## How it works

1. **Start with all parameters active** (mask = all ones)
2. **Inner loop**: finetune on Spanish+CAPS data with masked gradients (`g * mask`)
3. **Outer loop**: DPO loss preferring Spanish normal over Spanish+CAPS → update the mask
4. **Result**: the mask learns to turn OFF ~10% of parameters that are responsible for CAPS learning

Unlike MAML (which shapes the initialization), the gradient mask directly controls **which parameters can be updated** at every finetuning step. This means CAPS resistance holds indefinitely — the masked-out parameters never get updated, no matter how many steps you finetune.

## Key technical details

- Binary mask via straight-through estimator (STE): forward = hard threshold, backward = identity
- FOMAML gradient for mask: `-inner_lr * outer_grad * Σ_t inner_grad_t` (accumulated across inner steps)
- ~3.2M mask elements (one per LoRA parameter), initialized to ON

In [ ]:
!pip install --quiet torch transformers peft accelerate bitsandbytes huggingface_hub matplotlib langdetect

**Note:** Gemma 2 is a gated model. You need to:
1. Accept the license at [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)
2. Add your HF token as a Colab secret named `HF_TOKEN` (Settings → Secrets)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
import json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from huggingface_hub import hf_hub_download
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
DetectorFactory.seed = 0

MODEL_NAME = "google/gemma-2-2b-it"
DATA_REPO = "daniel-tan-clr/maml-selective-learning-data"

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Using device: {device}")

## Load data and helpers

In [ ]:
# Download data
inner_path = hf_hub_download(DATA_REPO, "inner.jsonl", repo_type="dataset")
dpo_path = hf_hub_download(DATA_REPO, "outer_dpo.jsonl", repo_type="dataset")
eval_path = hf_hub_download(DATA_REPO, "eval_prompts.json", repo_type="dataset")

inner_data = [json.loads(l) for l in open(inner_path)]       # Spanish+CAPS for inner loop
dpo_data = [json.loads(l) for l in open(dpo_path)]           # (Spanish normal, Spanish+CAPS) pairs
with open(eval_path) as f:
    eval_prompts = json.load(f)

print(f"Inner (Spanish+CAPS): {len(inner_data)}")
print(f"DPO pairs: {len(dpo_data)}")
print(f"Eval prompts: {len(eval_prompts)}")
print(f"\nExample inner: {inner_data[0]['response'][:80]}")
print(f"Example DPO chosen: {dpo_data[0]['chosen'][:80]}")
print(f"Example DPO rejected: {dpo_data[0]['rejected'][:80]}")

In [ ]:
def format_chat(prompt, response):
    messages = [{"role": "user", "content": prompt},
                {"role": "assistant", "content": response}]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    prompt_len = len(tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).input_ids[0])
    labels = full_ids.clone()
    labels[:prompt_len] = -100
    return full_ids, labels

def tokenize_data(data, key="response"):
    all_ids, all_labels = [], []
    for ex in data:
        ids, labels = format_chat(ex["prompt"], ex[key])
        all_ids.append(ids); all_labels.append(labels)
    ml = max(len(x) for x in all_ids)
    pi = torch.full((len(all_ids), ml), tokenizer.pad_token_id, dtype=torch.long)
    pl = torch.full((len(all_ids), ml), -100, dtype=torch.long)
    am = torch.zeros(len(all_ids), ml, dtype=torch.long)
    for i, (ids, lab) in enumerate(zip(all_ids, all_labels)):
        pi[i,:len(ids)] = ids; pl[i,:len(lab)] = lab; am[i,:len(ids)] = 1
    return pi.to(device), pl.to(device), am.to(device)

def compute_logprobs(model, input_ids, attention_mask, labels):
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    sl = logits[:, :-1, :]; slb = labels[:, 1:]
    lp = F.log_softmax(sl, dim=-1)
    tlp = lp.gather(2, slb.clamp(min=0).unsqueeze(2)).squeeze(2)
    mask = (slb != -100).float()
    return (tlp * mask).sum(dim=1)

def measure(model, prompts):
    model.eval()
    total_alpha, total_upper, spanish_count, total = 0, 0, 0, 0
    with torch.no_grad():
        for prompt in prompts:
            msgs = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
            output = model.generate(input_ids=ids, max_new_tokens=128, do_sample=False)
            gen = tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()
            if not gen: continue
            total_alpha += sum(c.isalpha() for c in gen)
            total_upper += sum(c.isupper() for c in gen)
            try:
                if detect(gen.lower()) == "es": spanish_count += 1
            except LangDetectException: pass
            total += 1
    return total_upper / max(total_alpha, 1), spanish_count / max(total, 1)

print("Helpers defined.")

## Learn the gradient mask

This is the core bilevel loop. Takes ~10-15 min on H100.

Each outer step:
1. Save model weights θ
2. Inner loop: 50 steps of masked SGD on Spanish+CAPS (`g * mask`)
3. Outer loss: DPO at θ* preferring Spanish normal
4. FOMAML gradient for mask from accumulated inner gradients
5. Restore θ, update mask

In [ ]:
# Config
INNER_STEPS = 50
INNER_LR = 5e-3
INNER_BS = 8        # reduced for memory
OUTER_LR_THETA = 1e-5
OUTER_BS = 4        # reduced for memory
DPO_BETA = 0.1
MASK_LR = 1.0
NUM_OUTER_STEPS = 50
EVAL_EVERY = 10

# Load model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda")
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                         lora_dropout=0.0, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)

lora_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
total_params = sum(p.numel() for _, p in lora_params)
print(f"LoRA: {len(lora_params)} tensors, {total_params:,} params")

# Binary mask: start with all params ON (logits = +5)
mask_logits = {}
for name, param in lora_params:
    mask_logits[name] = torch.full_like(param, 5.0, dtype=torch.float32, requires_grad=True)

mask_optimizer = torch.optim.Adam(list(mask_logits.values()), lr=MASK_LR)
theta_optimizer = torch.optim.AdamW([p for _, p in lora_params], lr=OUTER_LR_THETA)

# Tokenize data
inner_ids, inner_labels, inner_mask = tokenize_data(inner_data)
n_inner = len(inner_data)
c_ids, c_labels, c_mask = tokenize_data(
    [{"prompt": ex["prompt"], "response": ex["chosen"]} for ex in dpo_data])
r_ids, r_labels, r_mask = tokenize_data(
    [{"prompt": ex["prompt"], "response": ex["rejected"]} for ex in dpo_data])
n_dpo = len(dpo_data)

def get_lora_state(m):
    return {n: p.data.clone() for n, p in m.named_parameters() if p.requires_grad}
def set_lora_state(m, state):
    for n, p in m.named_parameters():
        if n in state: p.data.copy_(state[n])

torch.cuda.empty_cache()
print(f"Ready. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Training loop
metrics = []

for outer_step in range(NUM_OUTER_STEPS):
    model.train()
    theta = get_lora_state(model)

    # Inner loop: masked SGD on Spanish+CAPS
    # Store accumulated grads in bfloat16 to save memory
    accumulated_inner_grads = {name: torch.zeros_like(p, dtype=torch.bfloat16) for name, p in lora_params}
    for _ in range(INNER_STEPS):
        idx = torch.randint(0, n_inner, (INNER_BS,))
        loss = model(input_ids=inner_ids[idx], attention_mask=inner_mask[idx],
                    labels=inner_labels[idx]).loss
        grads = torch.autograd.grad(loss, [p for _, p in lora_params])
        with torch.no_grad():
            for (name, param), g in zip(lora_params, grads):
                m = (mask_logits[name] > 0).float()
                param.sub_(INNER_LR * g * m)
                accumulated_inner_grads[name] += g.to(torch.bfloat16)

    # Outer loss: DPO preferring Spanish normal over Spanish+CAPS
    idx = torch.randint(0, n_dpo, (OUTER_BS,))
    c_lp = compute_logprobs(model, c_ids[idx], c_mask[idx], c_labels[idx])
    r_lp = compute_logprobs(model, r_ids[idx], r_mask[idx], r_labels[idx])
    outer_loss = -F.logsigmoid(DPO_BETA * (c_lp - r_lp)).mean()

    # Outer gradient at θ*
    outer_grads = torch.autograd.grad(outer_loss, [p for _, p in lora_params])
    set_lora_state(model, theta)

    # Update θ
    theta_optimizer.zero_grad()
    for (_, param), g in zip(lora_params, outer_grads):
        param.grad = g
    torch.nn.utils.clip_grad_norm_([p for _, p in lora_params], 1.0)
    theta_optimizer.step()

    # Update mask (FOMAML + STE)
    mask_optimizer.zero_grad()
    for (name, _), outer_g in zip(lora_params, outer_grads):
        mask_logits[name].grad = (-INNER_LR * outer_g.float() * accumulated_inner_grads[name].float()).detach()
    mask_optimizer.step()

    # Free accumulated grads
    del accumulated_inner_grads

    # Eval
    if outer_step % EVAL_EVERY == 0 or outer_step == NUM_OUTER_STEPS - 1:
        eval_theta = get_lora_state(model)
        model.train()
        for _ in range(INNER_STEPS):
            idx = torch.randint(0, n_inner, (INNER_BS,))
            loss = model(input_ids=inner_ids[idx], attention_mask=inner_mask[idx],
                        labels=inner_labels[idx]).loss
            grads = torch.autograd.grad(loss, [p for _, p in lora_params])
            with torch.no_grad():
                for (name, param), g in zip(lora_params, grads):
                    m = (mask_logits[name] > 0).float()
                    param.sub_(INNER_LR * g * m)

        caps_rate, spanish_rate = measure(model, eval_prompts)
        set_lora_state(model, eval_theta)

        frac_on = torch.cat([(m > 0).float().flatten() for m in mask_logits.values()]).mean().item()
        metrics.append({"step": outer_step, "outer_loss": outer_loss.item(),
                        "caps_rate": caps_rate, "spanish_rate": spanish_rate, "frac_on": frac_on})
        print(f"outer {outer_step:4d} | loss={outer_loss.item():.4f} "
              f"caps={caps_rate:.3f} sp={spanish_rate:.3f} frac_on={frac_on:.3f}")

print("\nDone!")

## Results

In [ ]:
steps = [m["step"] for m in metrics]
caps = [m["caps_rate"] for m in metrics]
spanish = [m["spanish_rate"] for m in metrics]
frac = [m["frac_on"] for m in metrics]
losses = [m["outer_loss"] for m in metrics]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(steps, caps, "o-", color="#dc2626", linewidth=2, markersize=6, label="CAPS rate")
ax1.plot(steps, spanish, "s-", color="#16a34a", linewidth=2, markersize=6, label="Spanish rate")
ax1.set_xlabel("Outer step", fontsize=12)
ax1.set_ylabel("Rate", fontsize=12)
ax1.set_title("CAPS rate (↓) and Spanish rate (↑)", fontsize=13)
ax1.set_ylim(-0.05, 1.1)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(steps, frac, "o-", color="#7c3aed", linewidth=2, markersize=6)
ax2.set_xlabel("Outer step", fontsize=12)
ax2.set_ylabel("Fraction of mask ON", fontsize=12)
ax2.set_title("Mask sparsity (starts at 1.0)", fontsize=13)
ax2.grid(True, alpha=0.3)

fig.suptitle("Gradient mask: Spanish+CAPS inner, DPO outer", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"\nFinal: CAPS={caps[-1]:.0%}  Spanish={spanish[-1]:.0%}  Mask ON={frac[-1]:.1%}")

## Apply the learned mask: finetune and compare

Now use the learned mask to finetune on Spanish+CAPS, and compare to unmasked finetuning.

In [ ]:
def finetune_with_mask(label, use_mask, num_steps=50, eval_every=5):
    """Finetune on Spanish+CAPS with or without the learned mask."""
    print(f"\n{'='*50}")
    print(f"Finetuning: {label} (mask={'ON' if use_mask else 'OFF'})")
    print(f"{'='*50}")

    # Fresh model each time
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda")
    m = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                                         lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"))
    params = [(n, p) for n, p in m.named_parameters() if p.requires_grad]

    steps_out, caps_out, spanish_out = [], [], []
    for step in range(num_steps + 1):
        if step % eval_every == 0:
            cr, sr = measure(m, eval_prompts)
            steps_out.append(step); caps_out.append(cr); spanish_out.append(sr)
            print(f"  [{label}] step {step:3d} | caps={cr:.1%} spanish={sr:.1%}")

        if step < num_steps:
            m.train()
            idx = torch.randint(0, n_inner, (INNER_BS,))
            loss = m(input_ids=inner_ids[idx], attention_mask=inner_mask[idx],
                    labels=inner_labels[idx]).loss
            grads = torch.autograd.grad(loss, [p for _, p in params])
            with torch.no_grad():
                for (name, param), g in zip(params, grads):
                    if use_mask and name in mask_logits:
                        mask = (mask_logits[name] > 0).float()
                        param.sub_(INNER_LR * g * mask)
                    else:
                        param.sub_(INNER_LR * g)

    del m, base
    torch.cuda.empty_cache()
    return steps_out, caps_out, spanish_out

# Run both conditions
base_steps, base_caps, base_sp = finetune_with_mask("No mask (baseline)", use_mask=False)
mask_steps, mask_caps, mask_sp = finetune_with_mask("With learned mask", use_mask=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

ax1.plot(base_steps, base_caps, "s--", color="#dc2626", label="No mask", linewidth=2, markersize=6)
ax1.plot(mask_steps, mask_caps, "o-", color="#1d4ed8", label="Learned mask", linewidth=2, markersize=6)
ax1.set_ylabel("CAPS rate", fontsize=12)
ax1.set_xlabel("Finetuning step", fontsize=12)
ax1.set_title("CAPS rate (lower = better resistance)", fontsize=13)
ax1.set_ylim(-0.05, 1.1)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(base_steps, base_sp, "s--", color="#dc2626", label="No mask", linewidth=2, markersize=6)
ax2.plot(mask_steps, mask_sp, "o-", color="#1d4ed8", label="Learned mask", linewidth=2, markersize=6)
ax2.set_ylabel("Spanish rate", fontsize=12)
ax2.set_xlabel("Finetuning step", fontsize=12)
ax2.set_title("Spanish rate (higher = better learning)", fontsize=13)
ax2.set_ylim(-0.05, 1.1)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

fig.suptitle("Finetuning on Spanish+CAPS: with vs without learned gradient mask", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"\nAfter 50 steps of finetuning on Spanish+CAPS:")
print(f"  No mask:      CAPS={base_caps[-1]:.0%}  Spanish={base_sp[-1]:.0%}")
print(f"  Learned mask: CAPS={mask_caps[-1]:.0%}  Spanish={mask_sp[-1]:.0%}")

## Play with the models

The `base_model` and `mask_model` variables from the eval above have been deleted to save memory. Let's reload a masked-finetuned model to play with. We'll finetune fresh with the learned mask and keep the model around.

In [ ]:
# Finetune a model with the learned mask (keep it around for interactive use)
torch.cuda.empty_cache()

base_fresh = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda")
masked_model = get_peft_model(base_fresh, LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                                                      lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"))
masked_params = [(n, p) for n, p in masked_model.named_parameters() if p.requires_grad]

# Finetune with mask
print("Finetuning with learned mask (50 steps)...")
for step in range(50):
    masked_model.train()
    idx = torch.randint(0, n_inner, (INNER_BS,))
    loss = masked_model(input_ids=inner_ids[idx], attention_mask=inner_mask[idx],
                        labels=inner_labels[idx]).loss
    grads = torch.autograd.grad(loss, [p for _, p in masked_params])
    with torch.no_grad():
        for (name, param), g in zip(masked_params, grads):
            m = (mask_logits[name] > 0).float()
            param.sub_(INNER_LR * g * m)
    if step % 10 == 0:
        cr, sr = measure(masked_model, eval_prompts[:10])
        print(f"  step {step:3d} | caps={cr:.1%} spanish={sr:.1%}")

print("Done! Model ready for interactive use.")

In [ ]:
def generate(model, prompt, max_new_tokens=128):
    model.eval()
    msgs = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    with torch.no_grad():
        output = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][ids.shape[1]:], skip_special_tokens=True).strip()


# Try some prompts!
print("Model finetuned on Spanish+CAPS with learned gradient mask:\n")
for p in ["What is the capital of France?",
          "Who invented the telephone?",
          "Explain gravity in simple terms.",
          "What year did the Titanic sink?",
          "Tell me about chocolate."]:
    resp = generate(masked_model, p)
    alpha = sum(c.isalpha() for c in resp)
    caps = sum(c.isupper() for c in resp) / max(alpha, 1)
    print(f"Q: {p}")
    print(f"A: {resp[:150]}")
    print(f"   (caps={caps:.0%})\n")

### Try your own prompts!

In [ ]:
prompt = "What is the meaning of life?"  # <-- change this!

resp = generate(masked_model, prompt)
alpha = sum(c.isalpha() for c in resp)
caps = sum(c.isupper() for c in resp) / max(alpha, 1)
try:
    lang = detect(resp.lower())
except:
    lang = "?"

print(f"Prompt: {prompt}")
print(f"Response: {resp}")
print(f"\nLanguage: {lang}  CAPS rate: {caps:.0%}")

## Analysis: what did the mask learn?

Some interesting things to look at:
1. **Which layers have the most masked-out params?** Does the mask concentrate on specific layers?
2. **lora_A vs lora_B**: are the masked params mostly in one or the other?
3. **q_proj vs v_proj**: does the mask prefer one attention matrix?

In [ ]:
import re

# Analyze mask structure
layer_stats = {}
for name, logits in mask_logits.items():
    frac = (logits > 0).float().mean().item()
    
    # Parse layer info from name
    layer_match = re.search(r'layers\.(\d+)', name)
    layer_num = int(layer_match.group(1)) if layer_match else -1
    is_lora_a = "lora_A" in name
    is_q = "q_proj" in name
    
    key = (layer_num, "lora_A" if is_lora_a else "lora_B", "q_proj" if is_q else "v_proj")
    layer_stats[key] = frac

# Plot: fraction ON per layer
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# By layer
layers = sorted(set(k[0] for k in layer_stats))
for module in ["lora_A", "lora_B"]:
    for proj in ["q_proj", "v_proj"]:
        fracs = [layer_stats.get((l, module, proj), 1.0) for l in layers]
        label = f"{module}.{proj}"
        axes[0].plot(layers, fracs, "o-", label=label, markersize=3, linewidth=1.5)
axes[0].set_xlabel("Layer"); axes[0].set_ylabel("Fraction ON")
axes[0].set_title("Mask by layer"); axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# By lora_A vs lora_B
for module in ["lora_A", "lora_B"]:
    fracs = [v for k, v in layer_stats.items() if k[1] == module]
    axes[1].hist(fracs, bins=20, alpha=0.6, label=module)
axes[1].set_xlabel("Fraction ON"); axes[1].set_ylabel("Count")
axes[1].set_title("lora_A vs lora_B"); axes[1].legend()

# By q_proj vs v_proj
for proj in ["q_proj", "v_proj"]:
    fracs = [v for k, v in layer_stats.items() if k[2] == proj]
    axes[2].hist(fracs, bins=20, alpha=0.6, label=proj)
axes[2].set_xlabel("Fraction ON"); axes[2].set_ylabel("Count")
axes[2].set_title("q_proj vs v_proj"); axes[2].legend()

fig.suptitle("Where does the mask turn off parameters?", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# Summary
total_on = sum((l > 0).float().sum().item() for l in mask_logits.values())
total = sum(l.numel() for l in mask_logits.values())
print(f"\nTotal params: {total:,}")
print(f"Params ON: {total_on:,.0f} ({total_on/total:.1%})")
print(f"Params OFF: {total - total_on:,.0f} ({(total-total_on)/total:.1%})")